# Model evaluation

Per-tool peptide accuracy and FDR yields, before and after finetuning.

Set `VARIANT` to one of `pretrained`, `finetuned`, `comparison` and `SAMPLES` to one of `all`, `ecoli`, `wastewater` in the next cell, then run all cells.

- **Accuracy table** — per-tool peptide-match rate on the E. coli GT subset (mod-stripped, I=L equivalent).
- **FDR yields table** — number of target PSMs accepted at 1/5/10% FDR, read from `novoboard_out*/<sample>/<tool>.fdr.csv`.
- **Optional**: a final cell re-computes those FDR CSVs from the raw target/decoy NovoBoard inputs (target-decoy walk). Skip it if the CSVs already exist.

In [1]:
VARIANT = 'comparison'   # 'pretrained' | 'finetuned' | 'comparison'
SAMPLES = 'all'          # 'all' | 'ecoli' | 'wastewater'

In [2]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

from compare_predictions import LOADERS
from consensus_predict   import canonical
from mgf_index           import build_index_to_scan

ALL_TOOLS = ['novor', 'casanovo', 'instanovo', 'instanovoplus']
FT_TOOLS  = ['casanovo', 'instanovo', 'instanovoplus']
FDR_LEVELS = [0.01, 0.05, 0.10]

MGF_DIR = 'data_mgf/ecoli'
GT_DIR  = 'ground_truth/ecoli'

VARIANT_CFG = {
    'pretrained': {
        'pred_root':     'result_mgf',
        'novoboard_out': 'novoboard_out',
        'tools':         ALL_TOOLS,
        'ecoli_acc':     ['Ecoli_EV_1', 'Ecoli_EV_2'],
        'fdr_all':       ['ecoli', 'wastewater_Sample1', 'wastewater_Sample2'],
        'fdr_ecoli':     ['ecoli'],
        'fdr_waste':     ['wastewater_Sample1', 'wastewater_Sample2'],
    },
    'finetuned': {
        'pred_root':     'result_finetune_mgf',
        'novoboard_out': 'novoboard_out_finetune',
        'tools':         FT_TOOLS,
        'ecoli_acc':     ['Ecoli_EV_2'],
        'fdr_all':       ['Ecoli_EV_2', 'wastewater_Sample1', 'wastewater_Sample2'],
        'fdr_ecoli':     ['Ecoli_EV_2'],
        'fdr_waste':     ['wastewater_Sample1', 'wastewater_Sample2'],
    },
}

# Novor loader (the other 3 are in compare_predictions.LOADERS)
_NOVOR_MOD = {'(Cam)': '(+57.02)', '(O)': '(+15.99)',
              '(N)':   '(+0.98)',  '(P)': '(+79.97)'}

def load_novor(path):
    rows, header = [], None
    with open(path) as f:
        for raw in f:
            line = raw.rstrip('\n')
            if not line.strip():
                continue
            if line.startswith('#'):
                if header is None and 'id' in line and 'scanNum' in line:
                    header = [c.strip() for c in line.lstrip('#').split(',') if c.strip()]
                continue
            if header is None:
                continue
            r = dict(zip(header, [c.strip() for c in line.split(',')]))
            try:
                idx = int(r.get('id', '')) - 1
            except ValueError:
                continue
            pep = r.get('peptide', '')
            for k, v in _NOVOR_MOD.items():
                pep = pep.replace(k, v)
            try:
                score = float(r.get('score', 'nan'))
            except ValueError:
                score = float('nan')
            rows.append({'scan': idx, 'seq': pep, 'score': score})
    return pd.DataFrame(rows)

ALL_LOADERS = dict(LOADERS)
ALL_LOADERS['novor'] = load_novor
EXTS = {'casanovo': '.mztab', 'instanovo': '.csv',
        'instanovoplus': '.csv', 'novor': '.csv'}

In [3]:
def find_pred(tool, sample, pred_root):
    return Path(pred_root) / tool / 'ecoli' / f"{sample}{EXTS[tool]}"

def load_gt(sample):
    df = pd.read_csv(Path(GT_DIR) / f"{sample}.csv", usecols=['Scan', 'Peptide'])
    return df.rename(columns={'Scan': 'scan', 'Peptide': 'gt'})

def correct_scans(tool, sample, idx_to_scan, gt_canon, pred_root):
    p = find_pred(tool, sample, pred_root)
    if not p.exists():
        return set()
    df = ALL_LOADERS[tool](str(p))[['scan', 'seq']]
    df = df.dropna(subset=['scan']).drop_duplicates('scan', keep='first')
    df['scan'] = df['scan'].map(idx_to_scan)
    df = df.dropna(subset=['scan'])
    df['scan'] = df['scan'].astype(int)
    df['canon'] = df['seq'].map(canonical)
    return {int(s) for s, c in zip(df['scan'], df['canon'])
            if c and gt_canon.get(int(s)) == c}

def accuracy_row(sample, tools, pred_root):
    mgf = Path(MGF_DIR) / f"{sample}.mgf"
    if not mgf.exists():
        return None
    idx = build_index_to_scan(str(mgf))
    gt = load_gt(sample)
    gt_canon = {int(s): canonical(g) for s, g in zip(gt['scan'], gt['gt'])}
    per = {t: correct_scans(t, sample, idx, gt_canon, pred_root) for t in tools}
    out = {'gt_n': len(gt), 'best': len(set().union(*per.values())) if per else 0}
    for t in tools:
        out[t] = len(per[t])
    return out

def yield_at_fdr(fdr_csv, level):
    if not fdr_csv.exists():
        return None
    df = pd.read_csv(fdr_csv, usecols=['is_target', 'estimated_fdr'])
    return int(((df['estimated_fdr'] <= level) & df['is_target']).sum())

def fdr_row(sample, tool, novoboard_out):
    p = Path(novoboard_out) / sample / f"{tool}.fdr.csv"
    return [yield_at_fdr(p, lvl) for lvl in FDR_LEVELS]

def pct_str(num, den):
    return '  n/a' if not den else f"{100 * num / den:>5.1f}%"

def fmt_int(x):
    return '    -   ' if x is None else f"{x:>8}"

def fmt_dlt(a, b):
    return '    -   ' if a is None or b is None else f"{a - b:>+8d}"

TOOL_LABEL = {'casanovo': 'casa', 'instanovo': 'inst', 'instanovoplus': 'inst+', 'novor': 'novor'}

## Accuracy on E. coli GT

Recall = `#correct / #GT_scans`. Match rule: modifications stripped, I = L equivalent.

In [4]:
def acc_header_cols(tools):
    return '  '.join(f"{TOOL_LABEL[t]:>7}" for t in tools)

def acc_pct_line(label, row, tools):
    if row is None:
        return f"{label:<22} (no data)"
    cells = '  '.join(f"{pct_str(row.get(t, 0), row['gt_n']):>7}" for t in tools)
    return f"{label:<22} {row['gt_n']:>6}   {cells}    {pct_str(row['best'], row['gt_n']):>7}"

def acc_delta_line(label, ft, pre, tools):
    cells = []
    for t in tools:
        if t in ft and t in pre and ft['gt_n'] and pre['gt_n']:
            d = 100 * (ft[t] / ft['gt_n'] - pre[t] / pre['gt_n'])
            cells.append(f"{d:>+5.1f}pp")
        else:
            cells.append(f"{'-':>7}")
    if ft.get('best') is not None and pre.get('best') is not None and ft['gt_n'] and pre['gt_n']:
        d = 100 * (ft['best'] / ft['gt_n'] - pre['best'] / pre['gt_n'])
        best = f"{d:>+5.1f}pp"
    else:
        best = f"{'-':>7}"
    return f"  {label:<20} {'':>6}   {'  '.join(cells)}    {best}"

def print_accuracy(variant):
    print('=' * 90)
    print(f"Accuracy on E. coli GT — variant={variant}")
    print('=' * 90)
    tools_hdr = ALL_TOOLS if variant == 'comparison' else VARIANT_CFG[variant]['tools']
    print(f"\n{'sample':<22} {'GT_n':>6}   {acc_header_cols(tools_hdr)}    {'best':>7}")
    print('-' * 90)

    if variant in ('pretrained', 'finetuned'):
        cfg = VARIANT_CFG[variant]
        for s in cfg['ecoli_acc']:
            row = accuracy_row(s, cfg['tools'], cfg['pred_root'])
            print(acc_pct_line(s, row, tools_hdr))
        return

    pre, ft = VARIANT_CFG['pretrained'], VARIANT_CFG['finetuned']
    union = list(dict.fromkeys(pre['ecoli_acc'] + ft['ecoli_acc']))
    for s in union:
        pre_r = accuracy_row(s, pre['tools'], pre['pred_root']) if s in pre['ecoli_acc'] else None
        ft_r  = accuracy_row(s, ft['tools'],  ft['pred_root'])  if s in ft['ecoli_acc']  else None
        if pre_r is not None:
            print(acc_pct_line(s, pre_r, tools_hdr))
        if ft_r is not None:
            print(acc_pct_line(f"{s} (ft)", ft_r, tools_hdr))
        if pre_r is not None and ft_r is not None:
            print(acc_delta_line('Δ (ft − pre)', ft_r, pre_r, tools_hdr))

if SAMPLES in ('all', 'ecoli'):
    print_accuracy(VARIANT)
else:
    print('(accuracy table skipped — wastewater has no GT)')

Accuracy on E. coli GT — variant=comparison

sample                   GT_n     novor     casa     inst    inst+       best
------------------------------------------------------------------------------------------
Ecoli_EV_1               1131     29.3%    76.7%    78.5%    83.0%      87.0%
Ecoli_EV_2               1273     29.5%    74.8%    78.7%    82.3%      86.8%
Ecoli_EV_2 (ft)          1273      0.0%    77.5%    84.3%    85.8%      88.5%
  Δ (ft − pre)                        -   +2.7pp   +5.6pp   +3.5pp     +1.6pp


## FDR yields

Reads `novoboard_out*/<sample>/<tool>.fdr.csv` and counts target PSMs at each threshold.

In [5]:
def print_fdr(variant, samples):
    print('=' * 90)
    print(f"FDR yields — variant={variant}")
    print('=' * 90)
    print(f"\n{'sample':<24} {'tool':<14} {'@1%FDR':>10} {'@5%FDR':>10} {'@10%FDR':>10}")
    print('-' * 90)

    if variant in ('pretrained', 'finetuned'):
        cfg = VARIANT_CFG[variant]
        for s in samples:
            for t in cfg['tools']:
                ys = fdr_row(s, t, cfg['novoboard_out'])
                if ys[0] is None:
                    print(f"{s:<24} {t:<14} {'(n/a)':>10} {'(n/a)':>10} {'(n/a)':>10}")
                else:
                    print(f"{s:<24} {t:<14} {ys[0]:>10} {ys[1]:>10} {ys[2]:>10}")
            print()
        return

    pre, ft = VARIANT_CFG['pretrained'], VARIANT_CFG['finetuned']
    for s in samples:
        for t in ALL_TOOLS:
            pre_y = fdr_row(s, t, pre['novoboard_out']) if t in pre['tools'] else [None]*3
            ft_y  = fdr_row(s, t, ft['novoboard_out'])  if t in ft['tools']  else [None]*3
            if pre_y[0] is None and ft_y[0] is None:
                continue
            print(f"{s:<24} {t:<14} {fmt_int(pre_y[0])} {fmt_int(pre_y[1])} {fmt_int(pre_y[2])}")
            if ft_y[0] is not None:
                print(f"{s+' (ft)':<24} {t:<14} {fmt_int(ft_y[0])} {fmt_int(ft_y[1])} {fmt_int(ft_y[2])}")
                print(f"  {'Δ (ft − pre)':<22} {'':<14} "
                      f"{fmt_dlt(ft_y[0], pre_y[0])} {fmt_dlt(ft_y[1], pre_y[1])} {fmt_dlt(ft_y[2], pre_y[2])}")
        print()

cfg_for_samples = VARIANT_CFG['finetuned'] if VARIANT == 'comparison' else VARIANT_CFG[VARIANT]
fdr_samples = {
    'all':        cfg_for_samples['fdr_all'],
    'ecoli':      cfg_for_samples['fdr_ecoli'],
    'wastewater': cfg_for_samples['fdr_waste'],
}[SAMPLES]

print_fdr(VARIANT, fdr_samples)

FDR yields — variant=comparison

sample                   tool               @1%FDR     @5%FDR    @10%FDR
------------------------------------------------------------------------------------------
Ecoli_EV_2               casanovo           1141     2200     2557
Ecoli_EV_2 (ft)          casanovo            983     2307     2650
  Δ (ft − pre)                              -158     +107      +93
Ecoli_EV_2               instanovo          1154     3056     3665
Ecoli_EV_2 (ft)          instanovo          1026     3178     3755
  Δ (ft − pre)                              -128     +122      +90
Ecoli_EV_2               instanovoplus       617     2643     3567
Ecoli_EV_2 (ft)          instanovoplus        26     1395     2659
  Δ (ft − pre)                              -591    -1248     -908

wastewater_Sample1       novor               195     7185    10976
wastewater_Sample1       casanovo           1040     8888    11026
wastewater_Sample1 (ft)  casanovo            875     9668    1145

## Optional — re-compute FDR CSVs from raw target/decoy

Skip this cell if `novoboard_out*/<sample>/<tool>.fdr.csv` already exist.
Runs a target-decoy walk on `novoboard_in*/.../{target,decoy}/...csv` and writes the merged FDR table out the same way the NovoBoard notebooks did originally.

In [6]:
def calculate_fdr(target_csv, decoy_csv, fdr_csv_out=None):
    target = pd.read_csv(target_csv); target['is_target'] = True
    decoy  = pd.read_csv(decoy_csv);  decoy['is_target']  = False
    df = pd.concat([target, decoy], ignore_index=True)
    df = df.sort_values(['Score', 'is_target'], ascending=[False, False]).reset_index(drop=True)
    t_cum = df['is_target'].cumsum().to_numpy()
    d_cum = (~df['is_target']).cumsum().to_numpy()
    df['estimated_fdr'] = d_cum / np.where(t_cum == 0, 1, t_cum)
    if fdr_csv_out is not None:
        Path(fdr_csv_out).parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(fdr_csv_out, index=False)
    return {lvl: int(((df['estimated_fdr'] <= lvl) & df['is_target']).sum())
            for lvl in FDR_LEVELS}

def fdr_jobs(tool):
    """Yield (sample_label, variant, target_csv, decoy_csv, out_csv) per job."""
    yield ('ecoli/Ecoli_EV_2', 'pretrained',
           Path(f'novoboard_in/{tool}/target/ecoli/Ecoli_EV_2.csv'),
           Path(f'novoboard_in/{tool}/decoy/ecoli/Ecoli_EV_2.csv'),
           Path(f'novoboard_out/Ecoli_EV_2/{tool}.fdr.csv'))
    yield ('ecoli/Ecoli_EV_2', 'finetuned',
           Path(f'novoboard_in_finetune/{tool}/target/ecoli/Ecoli_EV_2.csv'),
           Path(f'novoboard_in_finetune/{tool}/decoy/ecoli/Ecoli_EV_2.csv'),
           Path(f'novoboard_out_finetune/Ecoli_EV_2/{tool}.fdr.csv'))
    for s in ['wastewater_Sample1', 'wastewater_Sample2']:
        yield (f'wastewater/{s}', 'pretrained',
               Path(f'novoboard_in_merged/{tool}/target/wastewater/{s}.csv'),
               Path(f'novoboard_in_merged/{tool}/decoy/wastewater/{s}.csv'),
               Path(f'novoboard_out/{s}/{tool}.fdr.csv'))
        yield (f'wastewater/{s}', 'finetuned',
               Path(f'novoboard_in_finetune_merged/{tool}/target/wastewater/{s}.csv'),
               Path(f'novoboard_in_finetune_merged/{tool}/decoy/wastewater/{s}.csv'),
               Path(f'novoboard_out_finetune/{s}/{tool}.fdr.csv'))

RECOMPUTE_TOOLS = FT_TOOLS  # novor's FDR CSVs were produced by the original NovoBoard notebooks
for tool in RECOMPUTE_TOOLS:
    print(f"\n=== {tool} ===")
    for label, variant, tgt, dcy, out in fdr_jobs(tool):
        if not tgt.exists() or not dcy.exists():
            print(f"  SKIP {label}/{variant}: missing input ({tgt.name} or {dcy.name})")
            continue
        counts = calculate_fdr(tgt, dcy, fdr_csv_out=out)
        cs = '  '.join(f"@{int(l*100):>2}%={counts[l]:>5}" for l in FDR_LEVELS)
        print(f"  {label:<32} {variant:<12}  {cs}  -> {out}")


=== casanovo ===
  ecoli/Ecoli_EV_2                 pretrained    @ 1%= 1141  @ 5%= 2200  @10%= 2557  -> novoboard_out/Ecoli_EV_2/casanovo.fdr.csv
  ecoli/Ecoli_EV_2                 finetuned     @ 1%=  983  @ 5%= 2307  @10%= 2650  -> novoboard_out_finetune/Ecoli_EV_2/casanovo.fdr.csv
  wastewater/wastewater_Sample1    pretrained    @ 1%= 1040  @ 5%= 8888  @10%=11026  -> novoboard_out/wastewater_Sample1/casanovo.fdr.csv
  wastewater/wastewater_Sample1    finetuned     @ 1%=  875  @ 5%= 9668  @10%=11451  -> novoboard_out_finetune/wastewater_Sample1/casanovo.fdr.csv
  wastewater/wastewater_Sample2    pretrained    @ 1%= 1907  @ 5%= 6207  @10%= 7670  -> novoboard_out/wastewater_Sample2/casanovo.fdr.csv
  wastewater/wastewater_Sample2    finetuned     @ 1%= 1601  @ 5%= 6628  @10%= 7958  -> novoboard_out_finetune/wastewater_Sample2/casanovo.fdr.csv

=== instanovo ===
  ecoli/Ecoli_EV_2                 pretrained    @ 1%= 1154  @ 5%= 3056  @10%= 3665  -> novoboard_out/Ecoli_EV_2/instanovo.f